# Deliverable 3: Our First Model, Classification with KNN

**MOD10 Introduction to Machine Learning**, November 2026

- **Group:** CUPILLARD Eliot, DENOUAL Romain, ROY-NOUGUIER Erwann
- **Dataset:** Arbres publics sur le territoire de la Ville (consolidated file), City of Montreal, licence CC BY 4.0, same unchanged copy as Deliverable 2 (https://doi.org/10.5281/zenodo.23540888)
- **Category (`y`):** trunk size class (young, medium, mature) built from `DHP`, the trunk diameter at 1.4 m
- **Question:** Can KNN tell the size class of a public tree from its species, location and age?
- **Revision (November 2026):** after the checkpoint feedback, section 8 checks why k = 1 and k = 5 give almost the same accuracy. Sections 1 to 7 keep the same data, split and results; the k results now have 4 decimals.

## 0. Setup and data from Deliverable 2

We reuse the data saved at the end of Deliverable 2 (`data/processed/trees_encoded.csv`): cleaned and encoded, not scaled. We do not clean or encode again. The file comes from the `to_csv` line of section 3 of that notebook and is kept in the repository.

`X` is the table without `DHP`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_PATH = Path("../data/processed/trees_encoded.csv")

assert DATA_PATH.is_file(), "File missing: run Deliverable 2 with the to_csv line of section 3 not commented"
data = pd.read_csv(DATA_PATH, index_col="raw_row")
assert data.shape == (334950, 62), "Wrong shape: this is not the file of Deliverable 2"

X = data.drop(columns=["DHP"])
print("Data:", data.shape, "| Missing values:", data.isna().sum().sum())

Data: (334950, 62) | Missing values: 0


## 1. Our category

Our target `DHP` is a number and KNN needs a category, so we build one from `DHP`: the size class of the trunk.

| Class | DHP |
|---|---|
| `young` | up to 10 |
| `medium` | above 10, up to 30 |
| `mature` | above 30 |

- The cut points are round values close to the thirds of the data (7 and 32). The choice is ours, not an official scale.
- A DHP of exactly 10 is `young`, exactly 30 is `medium` (6,107 and 4,777 trees).
- `DHP` is a diameter, not an age: `mature` means a large trunk.
- 70% of the `young` trees have a DHP of exactly 5, probably a default value (see Deliverable 2).

In [2]:
y = pd.cut(data["DHP"], bins=[0, 10, 30, 250], labels=["young", "medium", "mature"])
data["DHP"].groupby(y, observed=True).agg(["count", "min", "median", "max"])

,count,min,median,max
DHP,,,,
young,129358,1.0,5.0,10.0
medium,89385,10.1,21.0,30.0
mature,116207,30.1,47.0,246.0


## 2. Features and label

`y` is the size class and `X` the encoded table without `DHP`: the category comes from it, so keeping it would be data leakage. The identifiers and the columns that repeat other information are not in `X` (see Deliverable 2).

The classes are fairly balanced. A model that always answers `young` is right 38.6% of the time: this is our baseline.

In [3]:
print("X:", X.shape, "| y:", y.shape)
print("Columns that are not numbers:", list(X.select_dtypes(exclude="number").columns))
print("Missing values:", X.isna().sum().sum())
pd.DataFrame({"trees": y.value_counts(), "share": y.value_counts(normalize=True).round(3)}).sort_index()

X: (334950, 61) | y: (334950,)
Columns that are not numbers: []
Missing values: 0


,trees,share
DHP,,
young,129358,0.386
medium,89385,0.267
mature,116207,0.347


## 3. Split and scale

We reuse the split of Deliverable 2 (`test_size=0.2`, `random_state=SEED`, no `stratify`), so train and test have the same rows. The class shares are the same in both (below). Deliverable 2 saved the data before the split, so we split and scale again here.

Because KNN uses distances, we standardize the same three columns as in Deliverable 2 (`Latitude`, `Longitude`, `age`), with the scaler fitted on the training set only.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Class shares in train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Class shares in test:", y_test.value_counts(normalize=True).round(3).to_dict())

Train: (267960, 61) | Test: (66990, 61)
Class shares in train: {'young': 0.386, 'mature': 0.347, 'medium': 0.267}
Class shares in test: {'young': 0.386, 'mature': 0.347, 'medium': 0.267}


In [5]:
columns_to_scale = ["Latitude", "Longitude", "age"]
scaler = StandardScaler()
# fit on the training set only
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

print("Mean and standard deviation after scaling:")
display(X_train[columns_to_scale].agg(["mean", "std"]).round(3))
display(X_test[columns_to_scale].agg(["mean", "std"]).round(3))

Mean and standard deviation after scaling:


,Latitude,Longitude,age
mean,0.0,-0.0,-0.0
std,1.0,1.0,1.0


,Latitude,Longitude,age
mean,0.003,0.003,-0.004
std,0.998,0.998,0.994


## 4. Train KNN and choose k

We compare the values of k on a validation set (20% of the training set), not on the test set as in class: if we kept the k with the best test score, the test score would be too optimistic. The test set is only used at the end.

The course suggests starting near the square root of the number of training rows (about 517). We try values from 1 to 517, at about a minute each. The scaler was fitted on the whole training set, validation rows included.

In [6]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)
print("Fit:", X_fit.shape, "| Validation:", X_val.shape)

predictions = {}
for k in [1, 5, 9, 15, 51, 151, 517]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_fit, y_fit)
    predictions[k] = model.predict(X_val)
    acc = accuracy_score(y_val, predictions[k])
    print(f"k={k} | accuracy={acc:.4f}")

Fit: (214368, 61) | Validation: (53592, 61)


k=1 | accuracy=0.8266


k=5 | accuracy=0.8267


k=9 | accuracy=0.8214


k=15 | accuracy=0.8120


k=51 | accuracy=0.7855


k=151 | accuracy=0.7655


k=517 | accuracy=0.7488


| k | 1 | 5 | 9 | 15 | 51 | 151 | 517 |
|---|---|---|---|---|---|---|---|
| Accuracy on the validation set | 0.8266 | 0.8267 | 0.8214 | 0.8120 | 0.7855 | 0.7655 | 0.7488 |

We keep k = 5. k = 1 and k = 5 are almost equal (0.8266 and 0.8267), then the accuracy goes down. We prefer k = 5 because with k = 1 the answer depends on a single tree. Section 8 checks this result.

## 5. Evaluation

We train on the whole training set with k = 5 and test once on the test set. The rows of the matrix are the real classes, the columns the predicted ones. The baseline always answers `young`.

In [7]:
best_k = 5
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"baseline (always young): {(y_test == 'young').mean():.3f}")

labels = ["young", "medium", "mature"]
print("confusion matrix (rows = real class, columns = predicted class):")
print(pd.DataFrame(confusion_matrix(y_test, preds, labels=labels), index=labels, columns=labels))

accuracy: 0.837
baseline (always young): 0.386
confusion matrix (rows = real class, columns = predicted class):
        young  medium  mature
young   23593    1760     514
medium   1602   12590    3705
mature    235    3116   19875


## 6. Interpretation

The accuracy on the test set is 0.837, against 0.386 for the baseline.

10,932 trees out of 66,990 are wrong (16%). Most mistakes are between neighbouring classes (10,183, or 93%). Only 749 trees (1%) go from `young` to `mature` or the other way.

`medium` is the weakest class: 70% of its trees are right, against 91% for `young` and 86% for `mature`. It is mixed with both neighbours, and the cut points are our choice.

We do not know which mistake costs more to the city. If the size is used to plan work on large trees, the worst one is a `mature` tree predicted smaller: 3,351 of 23,226 (14%).

The classes are not rare, so accuracy is a fair first number, but it hides the weakness of `medium`. The confusion matrix shows it.

The score is lower on a new area: on boroughs the model has not seen, the accuracy is about 0.68 (section 8.5).

## 7. Model fit

KNN reaches 0.84 against 0.39 for the baseline, but it is slow on 335,000 trees and our real target is a number, so a regression model is probably a better fit (see "Choosing your model" in the Class 4 slides).

## 8. Follow-up: k = 1 and k = 5 give almost the same accuracy

At the checkpoint, the instructor saw this as a red flag. We check it here. To keep the run short, we use samples with a fixed seed: 10,000 test trees per split, 2,000 per borough.

### 8.1 Exact numbers on the validation set

In [8]:
real = y_val.to_numpy()
print("Validation trees:", len(real))
print("Right with k=1:", (predictions[1] == real).sum(), "| right with k=5:", (predictions[5] == real).sum())
print("Trees where k=1 and k=5 give a different answer:", (predictions[1] != predictions[5]).sum())

Validation trees: 53592
Right with k=1: 44301 | right with k=5: 44303
Trees where k=1 and k=5 give a different answer: 6764


### 8.2 Other random splits

If the gap is only noise, it changes from one split to another.

In [9]:
def knn_accuracy(X_tr, y_tr, X_te, y_te, k):
    X_tr, X_te = X_tr.copy(), X_te.copy()
    scaler = StandardScaler()
    X_tr[columns_to_scale] = scaler.fit_transform(X_tr[columns_to_scale])
    X_te[columns_to_scale] = scaler.transform(X_te[columns_to_scale])
    model = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    return accuracy_score(y_te, model.predict(X_te))


for seed in [0, 1, 2, 3, 42]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    sample_index = X_te.sample(10000, random_state=seed).index
    acc_1 = knn_accuracy(X_tr, y_tr, X_te.loc[sample_index], y_te.loc[sample_index], 1)
    acc_5 = knn_accuracy(X_tr, y_tr, X_te.loc[sample_index], y_te.loc[sample_index], 5)
    print(f"seed={seed} | k=1: {acc_1:.4f} | k=5: {acc_5:.4f} | k=5 minus k=1: {acc_5 - acc_1:+.4f}")

seed=0 | k=1: 0.8322 | k=5: 0.8376 | k=5 minus k=1: +0.0054


seed=1 | k=1: 0.8336 | k=5: 0.8339 | k=5 minus k=1: +0.0003


seed=2 | k=1: 0.8341 | k=5: 0.8314 | k=5 minus k=1: -0.0027


seed=3 | k=1: 0.8302 | k=5: 0.8302 | k=5 minus k=1: +0.0000


seed=42 | k=1: 0.8373 | k=5: 0.8438 | k=5 minus k=1: +0.0065


### 8.3 More classes

The instructor suggested that 3 classes may be too few. We try 5 and 8 classes on the seed 42 split.

In [10]:
X_tr, X_te, _, _ = train_test_split(X, y, test_size=0.2, random_state=SEED)
sample_index = X_te.sample(10000, random_state=SEED).index

for bins in [[0, 10, 30, 250], [0, 5, 10, 20, 40, 250], [0, 5, 10, 15, 20, 30, 40, 60, 250]]:
    classes = pd.cut(data["DHP"], bins=bins, labels=False)
    acc_1 = knn_accuracy(X_tr, classes.loc[X_tr.index], X_te.loc[sample_index], classes.loc[sample_index], 1)
    acc_5 = knn_accuracy(X_tr, classes.loc[X_tr.index], X_te.loc[sample_index], classes.loc[sample_index], 5)
    print(f"{len(bins) - 1} classes | k=1: {acc_1:.4f} | k=5: {acc_5:.4f} | k=5 minus k=1: {acc_5 - acc_1:+.4f}")

3 classes | k=1: 0.8373 | k=5: 0.8437 | k=5 minus k=1: +0.0064


5 classes | k=1: 0.7439 | k=5: 0.7453 | k=5 minus k=1: +0.0014


8 classes | k=1: 0.6126 | k=5: 0.6161 | k=5 minus k=1: +0.0035


### 8.4 The nearest training tree

`kneighbors()` gives the closest training tree of each test tree. The distance in meters is approximate (1 degree is about 111,000 m for latitude and 78,000 m for longitude).

In [11]:
nearest = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)
sample = X_test.sample(10000, random_state=SEED)
_, position = nearest.kneighbors(sample)
neighbour = X_train.iloc[position[:, 0]]

species_columns = [c for c in X.columns if c.startswith("Essence_latin_")]
same_species = (neighbour[species_columns].to_numpy() == sample[species_columns].to_numpy()).all(axis=1)
same_age = (neighbour["age"].to_numpy() == sample["age"].to_numpy()) & (sample["age_missing"].to_numpy() == 0)
d_lat = (data.loc[neighbour.index, "Latitude"].to_numpy() - data.loc[sample.index, "Latitude"].to_numpy()) * 111_000
d_lon = (data.loc[neighbour.index, "Longitude"].to_numpy() - data.loc[sample.index, "Longitude"].to_numpy()) * 78_000
meters = pd.Series((d_lat**2 + d_lon**2) ** 0.5)

print(f"Same species: {same_species.mean():.1%} | same species and same known age: {(same_species & same_age).mean():.1%}")
print(f"Distance to the nearest training tree: median {meters.median():.0f} m | under 50 m: {(meters < 50).mean():.1%}")

Same species: 99.9% | same species and same known age: 52.8%
Distance to the nearest training tree: median 17 m | under 50 m: 67.6%


### 8.5 Boroughs the model has not seen

For each borough, we train on the 12 others and test on 2,000 trees of that borough. Trees of the same street can no longer be in train and test.

In [12]:
borough_columns = [c for c in X.columns if c.startswith("ARROND_NOM_")]
rows = []
for column in borough_columns:
    inside = X[column] == 1
    sample_index = X[inside].sample(2000, random_state=SEED).index
    rows.append({
        "borough": column.replace("ARROND_NOM_", ""),
        "k=1": knn_accuracy(X[~inside], y[~inside], X.loc[sample_index], y.loc[sample_index], 1),
        "k=5": knn_accuracy(X[~inside], y[~inside], X.loc[sample_index], y.loc[sample_index], 5),
        "always young": (y.loc[sample_index] == "young").mean(),
    })
boroughs = pd.DataFrame(rows)
print(f"Mean | k=1: {boroughs['k=1'].mean():.3f} | k=5: {boroughs['k=5'].mean():.3f} | k=5 better in {(boroughs['k=5'] > boroughs['k=1']).sum()} of {len(boroughs)} boroughs")
boroughs.round(3)

Mean | k=1: 0.650 | k=5: 0.683 | k=5 better in 13 of 13 boroughs


,borough,k=1,k=5,always young
0,Ahuntsic - Cartierville,0.720,0.752,0.359
1,Côte-des-Neiges - Notre-Dame-de-Grâce,0.680,0.716,0.367
2,Lachine,0.292,0.306,0.200
3,Le Plateau-Mont-Royal,0.699,0.732,0.352
4,Le Sud-Ouest,0.730,0.739,0.430
5,Mercier - Hochelaga-Maisonneuve,0.763,0.796,0.438
6,Pierrefonds - Roxboro,0.516,0.612,0.456
7,Rivière-des-Prairies - Pointe-aux-Trembles,0.622,0.666,0.314
8,Rosemont - La Petite-Patrie,0.696,0.737,0.354
9,Saint-Léonard,0.676,0.714,0.372


### 8.6 Conclusion

- k = 1 and k = 5 are two different models: they disagree on 6,764 validation trees, but get almost the same number right (44,301 and 44,303).
- On a random split, the gap changes from one split to another (from −0.0027 to +0.0065). It is noise.
- More classes lower the accuracy (about 0.84, 0.74 and 0.61 for 3, 5 and 8 classes), but k = 1 and k = 5 stay close. The number of classes does not explain it.
- The cause is the nearest training tree: it is often a tree planted with the test tree (median 17 m away, same species for 99.9%, same species and same known age for 52.8%). With k = 1, the model copies it.
- On unseen boroughs, this neighbour is gone: the accuracy drops to 0.650 (k = 1) and 0.683 (k = 5), and k = 5 is better in all 13 boroughs.

So we keep k = 5. The test accuracy of 0.837 is too high for a new area, where about 0.68 is a fairer number.